# 00 · Environment and paths

Code for **"Integrating Histology and Morphometrics for Deep Learning-Based Assessment of
Skeletal Muscle Injury."** This notebook checks the environment and shows which data the
other notebooks will use.

| notebook | what it does | data |
|---|---|---|
| 01 | builds and verifies the class-balanced dataset (`Dataset_balanced`) | images |
| 02 | ResNet-50, ResNet-50 + morphometrics and the morphometric-only MLP | images |
| 03 | logistic regression: animal-grouped CV tuning, then the test-set refit | images |
| 04 | XGBoost, image only and + morphometrics | images |
| 05 | zero-shot transfer of the rat ResNet-50 to human tissue | images |
| 06 | **every number in the manuscript** and Table 2, from the saved predictions | none |
| 07 | **Figures 1–4**, from the saved predictions | none |
| 08 | reloads each trained model and checks it reproduces its predictions | images + weights |
| 09 | the whole pipeline end to end | images |

**The histology images and trained weights are not distributed.** Notebooks 06 and 07
reproduce the manuscript's numbers and figures from the saved per-patch predictions in
`models/`. The others run the same code on a **synthetic dataset**, generated here on
first use by `scripts/make_synthetic_data.py` (≈ 2 min): synthetic laminin-stained muscle
sections in the layout of the real data. Their results describe synthetic images, not the
manuscript. With the original data in `data/`, the same notebooks use it instead.

Paths come from `scripts/paths.py` and can be overridden with environment variables
before Jupyter starts (`VML_DATA`, `VML_HUMAN_ROOT`, `VML_HUMAN_WORKBOOK`,
`VML_RESULTS`, `VML_WORK`).

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
# The histology images are not distributed. Without them this notebook runs on the
# synthetic dataset (scripts/make_synthetic_data.py) and writes to runs/synthetic/.
SYNTHETIC = (os.environ.get("VML_SYNTHETIC") == "1"          # forced, or no original data
             or not ((HERE / "data" / "Dataset_balanced" / "dataset.yaml").is_file() or "VML_DATA" in os.environ))
if SYNTHETIC:
    os.environ["VML_SYNTHETIC"] = "1"
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
if SYNTHETIC:   # generate and balance the synthetic data once (both steps skip existing output)
    subprocess.run([PY, str(HERE / "scripts" / "make_synthetic_data.py"), "--out", str(P.DATASETS)], check=True)
    subprocess.run([PY, str(HERE / "scripts" / "build_dataset.py"), "--only", "Dataset_balanced"],
                   check=True, capture_output=True)
print("mode           :", "SYNTHETIC data (not the manuscript's)" if SYNTHETIC else "original data")
print("dataset        :", P.DATA)
print("human data     :", P.HUMAN_ROOT)
print("results (used) :", P.RESULTS)

In [ ]:
import importlib, platform
for m in ["torch", "torchvision", "sklearn", "xgboost", "pandas", "numpy", "matplotlib", "omegaconf", "openpyxl"]:
    try:
        mod = importlib.import_module(m)
        print(f"{m:12s} {getattr(mod, '__version__', '?')}")
    except ImportError as e:
        print(f"{m:12s} MISSING ({e})")
print("python      ", platform.python_version())
import torch
print("CUDA        ", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
checks = {
    "dataset used by 01-05, 08, 09": P.DATA / "dataset.yaml",
    "human comparison folders": P.HUMAN_ROOT / "uninjured_1mo",
    "human morphometrics workbook": P.HUMAN_WB,
    "manuscript numbers (frozen)": P.REFERENCE / "manuscript_numbers.json",
    "manuscript dataset report": P.REFERENCE / "dataset_report.json",
    "LR CV table (manuscript)": P.REFERENCE / "lr_tuning_cv.csv",
    "saved predictions (models/)": P.ROOT / "models" / "neural_networks" / "cnn__control_vml" / "evaluation" / "predictions.csv",
}
for name, p in checks.items():
    print(f"{'ok ' if p.exists() else 'MISSING'}  {name:32s} {p.relative_to(P.ROOT) if P.ROOT in p.parents else p}")